# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [348]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"          # PREENCHER: nome do arquivo
PROCESSED = Path("..") / "data" / "processed"
TARGET = "STATUS"                                            # PREENCHER: variável alvo

pd.set_option("display.max_columns", None)

In [349]:
df = pd.read_csv(RAW)
df.head()

,ID,MONTHS_BALANCE,STATUS,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,0,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008804,-1,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008804,-2,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
3,5008804,-3,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
4,5008804,-4,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0


## 1. Dados faltantes

In [350]:
# Identificando colunas com valores nulos
df.isnull().sum()

ID                          0
MONTHS_BALANCE              0
STATUS                      0
CODE_GENDER                 0
FLAG_OWN_CAR                0
FLAG_OWN_REALTY             0
CNT_CHILDREN                0
AMT_INCOME_TOTAL            0
NAME_INCOME_TYPE            0
NAME_EDUCATION_TYPE         0
NAME_FAMILY_STATUS          0
NAME_HOUSING_TYPE           0
DAYS_BIRTH                  0
DAYS_EMPLOYED               0
FLAG_MOBIL                  0
FLAG_WORK_PHONE             0
FLAG_PHONE                  0
FLAG_EMAIL                  0
OCCUPATION_TYPE        240048
CNT_FAM_MEMBERS             0
dtype: int64

***Tratando os nulos da coluna OCCUPATION_TYPE***

In [351]:
# Pelo dicionário de dados, se o número na coluna DAYS_EMPLOYED for positivo, significa que a pessoa está desempregada.
# Logo, para essa condição, podemos substituir os nulos da coluna OCCUPATION_TYPE por "Unemployed".

for item in df[df.DAYS_EMPLOYED > 0].index:
  df.loc[item, "OCCUPATION_TYPE"] = "Unemployed"


# Já para o restante dos nulos, podemos substituir por "Unknown", pois não temos informações sobre a ocupação dessas pessoas.
df.fillna('Unknown', inplace=True)

df.OCCUPATION_TYPE.isnull().sum()

np.int64(0)

_________________
**Decisão:** 

Baseado no diconário de dados, se o número na coluna DAYS_EMPLOYED for positivo, significa que a pessoa está desempregada.

Logo, para essa condição, decidimos substituir os nulos da coluna OCCUPATION_TYPE por "Unemployed".

Posteriormente, o restante dos dados nulos na coluna OCCUPATION_TYPE foi substituído por "Unkown", pois esses dados não foram informados.
_________________

## 2. Definição da variável alvo

___________
A coluna **TARGET** será a coluna **COMPLIANCE** que será gerada a partir da análise do STATUS do cliente
____

In [352]:
# Agrupando df por cliente

cliente = df.groupby('ID')['STATUS'].sum().reset_index()
cliente.head()

,ID,STATUS
0,5008804,CCCCCCCCCCCCC10X
1,5008805,CCCCCCCCCCCC10X
2,5008806,CCCCCCCX00X0X0XXXXXX0XXX0XXX0X
3,5008808,0X0XX
4,5008809,XXXXX


_____________
**Para a análise será ultilizado o seguinte critério:**

Mal Pagador (1): clientes que apresentam os indicadores 2, 3, 4 ou 5 em seu STATUS, demonstrando inadimplência grave, com atrasos maiores de 60 dias.

Bom Pagador(0): clientes que apresentam os indicadores 0, 1, C ou X.
____

In [353]:
# Separando o status que identifica ma pagador (>=60 dias de atraso)
bad_status = {'2', '3', '4', '5'}

In [354]:
# Criando de uma coluna chamada COMPLIANCE que será preenchida com 0 ou 1
cliente['COMPLIANCE'] = cliente.STATUS.apply(lambda s: 1 if any(char in bad_status for char in str(s)) else 0)

# Excluindo da coluna STATUS, que não será mais utilizada
cliente.drop(columns='STATUS', inplace=True)

cliente.COMPLIANCE.unique()

array([0, 1])

In [355]:
# Juntando os 2 dataframes
df = pd.merge(df, cliente, on='ID', how='inner')
df.head()

,ID,MONTHS_BALANCE,STATUS,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE
0,5008804,0,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0
1,5008804,-1,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0
2,5008804,-2,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0
3,5008804,-3,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0
4,5008804,-4,C,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0


__________
Verificando as colunas que podem ser binarizadas
_____

In [356]:
# Verificando os valores únicos de cada coluna
for col in df.columns:
  if df[col].nunique() == 2:
    print(f"Coluna: {col}, Valores únicos: {df[col].unique()}")

Coluna: CODE_GENDER, Valores únicos: <StringArray>
['M', 'F']
Length: 2, dtype: str
Coluna: FLAG_OWN_CAR, Valores únicos: <StringArray>
['Y', 'N']
Length: 2, dtype: str
Coluna: FLAG_OWN_REALTY, Valores únicos: <StringArray>
['Y', 'N']
Length: 2, dtype: str
Coluna: FLAG_WORK_PHONE, Valores únicos: [1 0]
Coluna: FLAG_PHONE, Valores únicos: [0 1]
Coluna: FLAG_EMAIL, Valores únicos: [0 1]
Coluna: COMPLIANCE, Valores únicos: [0 1]


As colunas CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY, FLAG_WORK_PHONE, FLAG_PHONE, e FLAG_EMAIL possuem apenas 2 valores, porém apenas 3 delas (CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY) não são binarizadas.

Para binarizá-las, será utilizado o Labe Encoder.

In [357]:
# Separando as colunas
colunas = ['CODE_GENDER','FLAG_OWN_CAR','FLAG_OWN_REALTY']

# Aplicando o label_encorder
label_encoder = LabelEncoder()
for i in colunas:
  df[i] = label_encoder.fit_transform(df[i])
df.head()

,ID,MONTHS_BALANCE,STATUS,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE
0,5008804,0,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0
1,5008804,-1,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0
2,5008804,-2,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0
3,5008804,-3,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0
4,5008804,-4,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,0


Legenda: M = 1, F = 0, Y = 1, N = 0

In [358]:
# Excluindo a coluna FLAG_MOBIL que contém apenas 1 valor e não será interessante para a análise

df.drop(columns='FLAG_MOBIL', inplace = True)

In [359]:
df.head()

,ID,MONTHS_BALANCE,STATUS,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE
0,5008804,0,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2.0,0
1,5008804,-1,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2.0,0
2,5008804,-2,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2.0,0
3,5008804,-3,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2.0,0
4,5008804,-4,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2.0,0


Aplicando o Hot Encoder nas colunas categóricas

In [360]:
# Separando as colunas categóricas
categoricas = []

for i in df.columns:
  if df[i].dtype == 'str':
    categoricas.append(i)


# Aplicando o hot encoder
hot = []

for i in df.columns:
  hot = pd.get_dummies(df[categoricas], prefix = 'hot')


# Mesclando os  dataframes
df = pd.concat([df, hot], axis=1)
df.head(2)

,ID,MONTHS_BALANCE,STATUS,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,COMPLIANCE,hot_0,hot_1,hot_2,hot_3,hot_4,hot_5,hot_C,hot_X,hot_Commercial associate,hot_Pensioner,hot_State servant,hot_Student,hot_Working,hot_Academic degree,hot_Higher education,hot_Incomplete higher,hot_Lower secondary,hot_Secondary / secondary special,hot_Civil marriage,hot_Married,hot_Separated,hot_Single / not married,hot_Widow,hot_Co-op apartment,hot_House / apartment,hot_Municipal apartment,hot_Office apartment,hot_Rented apartment,hot_With parents,hot_Accountants,hot_Cleaning staff,hot_Cooking staff,hot_Core staff,hot_Drivers,hot_HR staff,hot_High skill tech staff,hot_IT staff,hot_Laborers,hot_Low-skill Laborers,hot_Managers,hot_Medicine staff,hot_Private service staff,hot_Realty agents,hot_Sales staff,hot_Secretaries,hot_Security staff,hot_Unemployed,hot_Unknown,hot_Waiters/barmen staff
0,5008804,0,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2.0,0,False,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,5008804,-1,C,1,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2.0,0,False,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False


## 3. Normalização / padronização

**Escolha e justificativa:** _StandardScaler, MinMaxScaler ou nenhum?_

Aplique preferencialmente dentro de um `Pipeline` no notebook 03 — ajustar o scaler antes do split vaza informação do teste para o treino.

In [361]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 777715 entries, 0 to 777714
Data columns (total 69 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   ID                                 777715 non-null  int64  
 1   MONTHS_BALANCE                     777715 non-null  int64  
 2   STATUS                             777715 non-null  str    
 3   CODE_GENDER                        777715 non-null  int64  
 4   FLAG_OWN_CAR                       777715 non-null  int64  
 5   FLAG_OWN_REALTY                    777715 non-null  int64  
 6   CNT_CHILDREN                       777715 non-null  int64  
 7   AMT_INCOME_TOTAL                   777715 non-null  float64
 8   NAME_INCOME_TYPE                   777715 non-null  str    
 9   NAME_EDUCATION_TYPE                777715 non-null  str    
 10  NAME_FAMILY_STATUS                 777715 non-null  str    
 11  NAME_HOUSING_TYPE                  777715 non-null

In [362]:
# Aplicando o hot encoder nas colunas categóricas




In [363]:
# escala

## 4. Feature engineering

**Justificativa:** _preencher._

In [364]:
# novas variáveis derivadas

## 5. Salvar dataset tratado

In [365]:
# PROCESSED.mkdir(parents=True, exist_ok=True)
# df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)